In [1]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


# Sample Images Extracted using Python

In [2]:
import os
import zipfile

# 1. Set the zip file path 
user_home = os.path.expanduser('~')

zip_path = os.path.join(user_home, 'Downloads', 'archive.zip') 

if not os.path.exists(zip_path):
    zip_path = os.path.join(user_home, 'Downloads', 'breast-histopathology-images.zip')

print(f"Zip file found at: {zip_path}")

# 2. Create folders
os.makedirs('data/0', exist_ok=True)
os.makedirs('data/1', exist_ok=True)

# 3. Only 2000-2000 images extract for easy to train and test
count_0 = 0
count_1 = 0
limit = 2000

print("Extracting 4,000 sample images (Please wait 1-2 minutes)...")

with zipfile.ZipFile(zip_path, 'r') as z:
    for file_info in z.infolist():
        filename = file_info.filename
        
        if filename.endswith('.png'):
            # Folder structure me 0 aur 1 check karein
            if '/0/' in filename and count_0 < limit:
                base_name = os.path.basename(filename)
                with open(os.path.join('data/0', base_name), 'wb') as f:
                    f.write(z.read(filename))
                count_0 += 1
            elif '/1/' in filename and count_1 < limit:
                base_name = os.path.basename(filename)
                with open(os.path.join('data/1', base_name), 'wb') as f:
                    f.write(z.read(filename))
                count_1 += 1
                
        if count_0 >= limit and count_1 >= limit:
            break

print(f"Done! Class 0: {count_0} images | Class 1: {count_1} images")

Zip file found at: C:\Users\SANDESH\Downloads\archive.zip
Extracting 4,000 sample images (Please wait 1-2 minutes)...
Done! Class 0: 2000 images | Class 1: 2000 images


# Data Loading & Split

In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Rescaling
from tensorflow.keras.utils import image_dataset_from_directory

IMG_SIZE = (50, 50)
BATCH_SIZE = 32

train_ds = image_dataset_from_directory(
    'data/',
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = image_dataset_from_directory(
    'data/',
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Sirf prefetch use karenge, cache() nahi taaki RAM bhar na jaye
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(500).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 4000 files belonging to 2 classes.
Using 3200 files for training.
Found 4000 files belonging to 2 classes.
Using 800 files for validation.


# Model Architecture (CancerNet) Define & Compile

In [5]:
model = Sequential([
    Rescaling(1./255, input_shape=(50, 50, 3)),
    
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5 Epochs pehle run karte hain verification ke liye
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5


C:\Users\SANDESH\anaconda3\envs\tensorflow\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\SANDESH\anaconda3\envs\tensorflow\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


100/100 ━━━━━━━━━━━━━━━━━━━━ 13s 93ms/step - accuracy: 0.8516 - loss: 0.5068 - val_accuracy: 0.0000e+00 - val_loss: 6.7439
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 20s 90ms/step - accuracy: 0.8778 - loss: 0.3418 - val_accuracy: 0.0000e+00 - val_loss: 8.8978
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 79ms/step - accuracy: 0.8928 - loss: 0.2984 - val_accuracy: 0.0000e+00 - val_loss: 8.1695
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 79ms/step - accuracy: 0.9034 - loss: 0.2886 - val_accuracy: 0.0012 - val_loss: 13.1521
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 89ms/step - accuracy: 0.9131 - loss: 0.2643 - val_accuracy: 0.8325 - val_loss: 0.7842


# Model Evaluation & Confusion Matrix

In [7]:
# Evaluate model directly
loss, acc = model.evaluate(val_ds, verbose=1)
print(f"\nFinal Validation Accuracy: {acc * 100:.2f}%")
print(f"Final Validation Loss: {loss:.4f}")

# Model save karein taaki GitHub par use ho sake
model.save("cancernet_model.keras")
print("Model successfully saved as cancernet_model.keras!")

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8325 - loss: 0.7842

Final Validation Accuracy: 83.25%
Final Validation Loss: 0.7842
Model successfully saved as cancernet_model.keras!


# Confusion Matrix & Metrics Display

In [9]:
import numpy as np

# 1. Prediction on validation data
y_true_list = []
y_pred_list = []

for images, labels in val_ds:
    preds = model(images, training=False).numpy().ravel()
    preds_binary = (preds > 0.5).astype(int)
    y_true_list.extend(labels.numpy().ravel())
    y_pred_list.extend(preds_binary)

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)

# 2. Confusion matrix counts calculate karna
tp = int(np.sum((y_true == 1) & (y_pred == 1)))
tn = int(np.sum((y_true == 0) & (y_pred == 0)))
fp = int(np.sum((y_true == 0) & (y_pred == 1)))
fn = int(np.sum((y_true == 1) & (y_pred == 0)))

total = tp + tn + fp + fn
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# 3. Formatted display
print("=" * 45)
print("       CANCERNET CONFUSION MATRIX       ")
print("=" * 45)
print(f"{'':<20} | Predicted Benign | Predicted Malignant")
print("-" * 55)
print(f"Actual Benign (0)    | {tn:<16} | {fp:<19}")
print(f"Actual Malignant (1) | {fn:<16} | {tp:<19}")
print("=" * 45)

print("\n--- FINAL CLASSIFICATION METRICS ---")
print(f"Overall Accuracy : {accuracy * 100:.2f}%")
print(f"Precision        : {precision:.4f}")
print(f"Recall           : {recall:.4f}")
print(f"F1-Score         : {f1:.4f}")
print("=" * 45)

       CANCERNET CONFUSION MATRIX       
                     | Predicted Benign | Predicted Malignant
-------------------------------------------------------
Actual Benign (0)    | 0                | 0                  
Actual Malignant (1) | 134              | 666                

--- FINAL CLASSIFICATION METRICS ---
Overall Accuracy : 83.25%
Precision        : 1.0000
Recall           : 0.8325
F1-Score         : 0.9086
